# Option 3 – split storage, local join (runs **on-premise**)

* **Snowflake** holds only non-sensitive attributes: `SOVEREIGNTY_DEMO.OPT3_SPLIT.EMPLOYEES_PUBLIC`.
* **On-premise** holds the sensitive columns (names, national IDs, IBANs, salaries) in the local Iceberg lake
  (Lakekeeper + S3-compatible storage – the same stack as Option 1, but **no tunnel is opened**).
* This notebook runs on an on-prem host. It pulls the *non-sensitive* rows from Snowflake, reads the sensitive
  rows locally with DuckDB and joins **in local memory**. Sensitive values never travel to Snowflake.
* Optionally, only **k-anonymous aggregates** are written back to Snowflake for broad consumption.

In [ ]:
import os, time
import duckdb, pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

SF_CONN = os.getenv("SNOW_CONNECTION", "default")
ICEBERG_REST = "http://localhost:8181/catalog/"
S3_ENDPOINT = "localhost:9000"
K_THRESHOLD = 5    # minimum group size before an aggregate may leave the premises

sf = snowflake.connector.connect(connection_name=SF_CONN, role="SOV_SPLIT_ROLE",
                                 warehouse="SOV_WH", database="SOVEREIGNTY_DEMO", schema="OPT3_SPLIT")
sf.cursor().execute("ALTER SESSION SET QUERY_TAG = 'sov_opt3_local_join'")
print("Snowflake session:", sf.cursor().execute("SELECT CURRENT_ACCOUNT(), CURRENT_USER(), CURRENT_ROLE()").fetchone())

## 1 · Pull the non-sensitive half from Snowflake

In [ ]:
public_df = sf.cursor().execute("SELECT * FROM EMPLOYEES_PUBLIC").fetch_pandas_all()
public_df.columns = [c.lower() for c in public_df.columns]
print(len(public_df), "rows from Snowflake"); public_df.head()

## 2 · Read the sensitive half from the on-prem Iceberg lake (never leaves this host)

In [ ]:
con = duckdb.connect(":memory:")
con.execute("INSTALL iceberg; LOAD iceberg; INSTALL httpfs; LOAD httpfs;")
con.execute(f"""CREATE SECRET onprem_s3 (TYPE S3, KEY_ID 'admin', SECRET 'password',
               ENDPOINT '{S3_ENDPOINT}', USE_SSL false, URL_STYLE 'path', REGION 'us-east-1')""")
con.execute("CREATE SECRET onprem_rest (TYPE ICEBERG, TOKEN 'dummy')")
con.execute(f"""ATTACH 'demo' AS lake (TYPE ICEBERG, ENDPOINT '{ICEBERG_REST}',
               SECRET onprem_rest, ACCESS_DELEGATION_MODE 'none')""")
sensitive_df = con.execute("SELECT * FROM lake.demo.employees_sensitive").fetchdf()
print(len(sensitive_df), "rows from on-prem lake"); sensitive_df.head(3)

## 3 · Join locally – full row-level view, only on this machine

In [ ]:
con.register("employees_public", public_df)
joined = con.execute("""
    SELECT p.employee_id, p.department, p.country, p.level, s.full_name, s.iban, s.salary, s.bonus
    FROM employees_public p JOIN lake.demo.employees_sensitive s USING (employee_id)
    ORDER BY s.salary DESC""").fetchdf()
joined.head(10)

In [ ]:
import matplotlib.pyplot as plt
agg = joined.groupby("department")["salary"].mean().sort_values()
agg.plot.barh(title="Average salary by department (computed on-prem)"); plt.xlabel("EUR"); plt.show()

## 4 · Optional: push back **only k-anonymous aggregates**

Groups smaller than `K_THRESHOLD` are suppressed, so no individual salary can be inferred in Snowflake.

In [ ]:
bands = con.execute(f"""
    SELECT p.department, p.country, COUNT(*) AS headcount,
           ROUND(AVG(s.salary)) AS avg_salary, ROUND(MEDIAN(s.salary)) AS p50_salary
    FROM employees_public p JOIN lake.demo.employees_sensitive s USING (employee_id)
    GROUP BY 1, 2 HAVING COUNT(*) >= {K_THRESHOLD} ORDER BY 1, 2""").fetchdf()
bands["computed_at"] = pd.Timestamp.now(tz="UTC")
bands["k_threshold"] = K_THRESHOLD
bands.columns = [c.upper() for c in bands.columns]
sf.cursor().execute("TRUNCATE TABLE SALARY_BANDS_AGG")
ok, _, n, _ = write_pandas(sf, bands, "SALARY_BANDS_AGG", use_logical_type=True)
print(f"wrote {n} k-anonymous aggregate rows (suppressed groups < {K_THRESHOLD})"); bands.head()

## 5 · What did Snowflake actually see? (its own query history for this session)

In [ ]:
time.sleep(2)
hist = sf.cursor().execute("""
    SELECT query_type, LEFT(query_text, 90) AS query_text, rows_produced
    FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION(RESULT_LIMIT => 50))
    WHERE query_tag = 'sov_opt3_local_join' ORDER BY start_time""").fetch_pandas_all()
hist

**Observation:** Snowflake only ever executed reads on `EMPLOYEES_PUBLIC` and writes of aggregated bands.
No query, result set, cache entry or stage file in Snowflake ever contained a name, IBAN or individual salary.
The trade-off: the joined data is outside Snowflake governance (RBAC, masking, lineage), and the join is limited
by local compute and memory.

In [ ]:
sf.close(); con.close()